# Case effects

**Purpose.** This notebook estimates, for every feasible treatment episode, the effect of the opening of a theme park or destination resort on international tourism receipts as a percentage of GDP in the host economy. An effect is the mean gap between the observed outcome of the host economy and a synthetic control built from donor economies, taken over the post-opening window. The standard error of an effect is the standard deviation of the placebo effects that arise when each donor is treated in turn as if it were the host. Where many different weightings reproduce the pre-opening path exactly, the notebook also reports the range of effects that all of them give. The notebook also sets the sample flags, runs robustness variants of the effect, and builds the tables of pre-opening features that notebook 03 uses to learn which features modify the effect.

**Input files.** `global_panel.csv` in the processed data directory and `panel_coverage.csv` and `donor_eligibility.csv` in the results directory, all written by notebook 00; the cases catalogue `cases_catalogue.csv`; and, in a real run, the file `cases_investment_fx.csv` with investment costs converted to US dollars. The first code cell prints the paths.

**Output files.** In the results directory: `episodes.csv`, `feasibility.csv`, `case_effects.csv`, `case_placebos.csv`, `case_effects_robustness.csv`, `reference_features.csv`, `case_features.csv`, `case_features_imputed.csv`, `imputation_report.csv` and `figure_inventory_02_case_effects.csv`. In the figures directory: `case_effects_forest.png`, `case_gap_paths.png`, `case_effect_vs_capex.png` and `case_estimator_comparison.png`.

**How to run.** On the desktop that holds the real data, set `DTT_RUN_REAL=1` and execute all cells, for example with `jupyter nbconvert --to notebook --execute notebooks/02_case_effects.ipynb`. To test the notebook on a simulated world, set `DTT_WORLD=sim` together with `DTT_SIM_DIR`, `DTT_RESULTS_DIR` and `DTT_FIGURE_DIR`, each pointing to a directory outside the repository, and run notebook 00 first. A simulated run refuses an output directory inside the data, results or figures directory of the repository, never reads the real files and never writes to the results directory of the repository.

**Run time.** Under one minute in a simulated run and a few minutes on the real data.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
_root = next(p for p in (_here, *_here.parents) if (p / "src" / "dtt" / "__init__.py").is_file())
sys.path.insert(0, str(_root / "src"))

from dtt import workflow

ctx = workflow.setup_run("02_case_effects")
ctx.describe()

## Settings

All settings are fixed here once and printed. The notebook has no repeat counts and no bootstrap sizes, so no setting depends on the mode. The seed of the effects stage is passed to every estimator call.

In [ ]:
import time
from itertools import pairwise

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.legend import Legend
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator
from matplotlib.transforms import Bbox

from dtt.cases import (
    CATALOGUE_COLUMNS,
    EPISODE_FIRST_YEAR,
    EPISODE_LAST_FEASIBLE_YEAR,
    FEATURE_COLUMNS,
    build_episodes,
    case_features,
    check_feasibility,
    donor_pool,
    economy_feature_cloud,
    impute_features,
    load_cases,
    outcome_matrix,
)
from dtt.effects import EXACT_FIT_TOLERANCE, estimate_all
from dtt.panel import read_panel

OUTCOME = "receipts_pct_gdp"
FIRST_YEAR, LAST_YEAR = 1995, 2019
MERGE_GAP = 5
POST_HORIZON = 5
MIN_POST = 3
SHORT_HORIZON, LONG_HORIZON = 3, 8
FEATURE_WINDOW = 3
REFERENCE_YEARS = range(FIRST_YEAR, LAST_YEAR + 1)
MAX_MISSING_SHARE = 0.25
RIDGE_FRACTIONS = {"ridge_low_penalty": 0.01, "ridge_high_penalty": 0.1}
FIT_FACTOR = 2.0
PLACEBO_INTERVAL = (0.05, 0.95)
MAX_PATH_EPISODES = 6
MIN_POINTS_FOR_LINE = 5
PARK_CATEGORIES = ("theme_park", "multi_park_resort")
SEED = ctx.seeds["effects"]

print(f"Outcome: {OUTCOME}; estimation window {FIRST_YEAR} to {LAST_YEAR}")
print(
    f"Episodes: from {EPISODE_FIRST_YEAR} on, an opening at most {MERGE_GAP} years after the previous opening of its economy joins that episode; "
    f"earlier openings are episodes of their own"
)
print(f"Catalogue flag panel_feasible of an episode: start year from {EPISODE_FIRST_YEAR} to {EPISODE_LAST_FEASIBLE_YEAR}")
print(f"Post-opening window: at most {POST_HORIZON} years; feasibility needs at least {MIN_POST} post-opening years")
print(f"Robustness horizons: {SHORT_HORIZON} and {LONG_HORIZON} years")
print(f"Features: mean of the {FEATURE_WINDOW} years before the opening; reference years {REFERENCE_YEARS.start} to {REFERENCE_YEARS.stop - 1}")
print(f"Features with a missing share above {MAX_MISSING_SHARE:.2f} are dropped")
print("Ridge penalties as fractions of the pre-opening scale: " + ", ".join(f"{k} {v}" for k, v in RIDGE_FRACTIONS.items()))
print(f"Park categories: {', '.join(PARK_CATEGORIES)}")
print(
    f"Pre-opening fit criterion: pre-opening RMSPE at most {FIT_FACTOR:g} times the median placebo value, "
    f"and not an exact fit with fewer pre-opening years than donors"
)
print(f"Exact fit: gap of at most {EXACT_FIT_TOLERANCE:g} percentage points of GDP in every pre-opening year")
print(f"Placebo interval of the effects figure: {100 * PLACEBO_INTERVAL[0]:.0f}th to {100 * PLACEBO_INTERVAL[1]:.0f}th percentile")
print(f"Gap-path figure: at most {MAX_PATH_EPISODES} episodes; investment line from {MIN_POINTS_FOR_LINE} episodes with a cost")
print(f"Seeds: effects {SEED}")

## Inputs

The panel is read from the file that notebook 00 wrote. The catalogue is read with the investment cost in US dollars that the analysis uses: the reported figure where the catalogue has one and the converted figure otherwise. The price year of a conversion and the flag that says whether it was assumed are kept as columns of the catalogue table.

In [ ]:
ctx.require("panel_coverage.csv", "donor_eligibility.csv")
panel_path = ctx.processed_dir / "global_panel.csv"
if not panel_path.is_file():
    raise FileNotFoundError(f"{panel_path} not found; run notebook 00_build_global_panel first")
panel = read_panel(panel_path)
catalogue = load_cases(ctx.catalogue_path, ctx.investment_fx_path)
assert list(catalogue.columns) == list(CATALOGUE_COLUMNS)
economies = sorted(panel["iso3"].unique())

print(f"Panel: {panel_path} with {len(panel)} rows, {len(economies)} economies")
print(f"Catalogue: {ctx.catalogue_path}")
print(f"Investment conversion file: {ctx.investment_fx_path}")
print(f"\n{len(catalogue)} cases in {catalogue['iso3'].nunique()} economies, opening years {catalogue['opening_year'].min()} to {catalogue['opening_year'].max()}")
print("\nCases by category:")
print(catalogue["category"].value_counts().rename("cases").to_string())
print("\nInvestment figure used, by basis:")
print(catalogue["investment_basis"].value_counts().rename("cases").to_string())
converted = (catalogue["investment_basis"] == "fx_converted").to_numpy()
assumed = catalogue["price_year_assumed"].fillna(False).astype(bool).to_numpy()
print(
    f"\nThe catalogue table has {catalogue.shape[1]} columns, including price_year and price_year_assumed. "
    f"{int(converted.sum())} investment figures are converted; the price year of {int((converted & assumed).sum())} of them is assumed."
)

## Episodes

Openings of one economy that follow each other within the merge gap are merged into one treatment episode, because the effect of a later opening cannot be separated from that of an earlier one when their post-opening windows overlap. Only openings from the first episode year of the settings cell on are merged. A chain of openings, each within the merge gap of the one before, forms a single episode, so an episode can span more than the merge gap. Each earlier opening is listed as an episode of its own and is not flagged as feasible in the catalogue. The unit of analysis is the episode. An episode is dated at its earliest opening, its investment is the sum of the known investments of its openings, and its category is that of the member with the largest investment. The table lists the investment of each member in the column `member_investment_usd_bn`. The catalogue flag `panel_feasible` of an episode needs a start year from the first episode year to the last feasible start year of the settings cell.

In [ ]:
episodes = build_episodes(catalogue, merge_gap=MERGE_GAP)
ctx.save_table(episodes, "episodes.csv")

member_ids = [str(t).split(";") for t in episodes["member_case_ids"]]
member_years = [[int(v) for v in str(t).split(";")] for t in episodes["member_opening_years"]]
member_money = [[float(v) if v.strip() else np.nan for v in str(t).split(";")] for t in episodes["member_investment_usd_bn"]]

merged = episodes.loc[episodes["n_openings"] > 1]
early = int((catalogue["opening_year"] < EPISODE_FIRST_YEAR).sum())
print(
    f"The catalogue has {len(catalogue)} cases in {catalogue['iso3'].nunique()} economies. "
    f"With a merge gap of {MERGE_GAP} years from {EPISODE_FIRST_YEAR} on they form {len(episodes)} episodes in {episodes['iso3'].nunique()} economies."
)
print(f"Openings before {EPISODE_FIRST_YEAR}, each listed as an episode of its own: {early}.")
print(f"Episodes that combine several openings: {len(merged)}, with {int(merged['n_openings'].sum())} openings in total.")
if len(merged):
    spans = [max(years) - min(years) for years in member_years if len(years) > 1]
    print(f"The longest episode has {int(merged['n_openings'].max())} openings and spans {max(spans)} years.")
    print(merged[["case_id", "n_openings", "member_opening_years", "member_investment_usd_bn", "investment_usd_bn_used"]].to_string(index=False))
flags = episodes["panel_feasible"].astype("object").fillna("not given")
print("\nCatalogue flag panel_feasible of the episodes: " + ", ".join(f"{k} {int(v)}" for k, v in flags.value_counts().items()))

assert episodes["case_id"].is_unique
assert int(episodes["n_openings"].sum()) == len(catalogue)
assert sorted(case for ids in member_ids for case in ids) == sorted(catalogue["case_id"])
assert all(len(a) == len(b) == len(c) == n for a, b, c, n in zip(member_ids, member_years, member_money, episodes["n_openings"], strict=True))
known_total = np.array([np.nansum(m) if np.isfinite(m).any() else np.nan for m in member_money])
assert np.allclose(known_total, episodes["investment_usd_bn_used"].to_numpy(dtype=float), rtol=1e-9, atol=0.0, equal_nan=True)
starts_by_economy = episodes.groupby("iso3")["opening_year"].apply(lambda s: sorted(set(s))).to_dict()
following = [next((y for y in starts_by_economy[i] if y > start), np.nan) for i, start in zip(episodes["iso3"], episodes["opening_year"], strict=True)]
assert np.allclose(episodes["next_start_year"].to_numpy(dtype=float), following, equal_nan=True)
for years in member_years:
    if len(years) > 1:
        assert min(years) >= EPISODE_FIRST_YEAR and all(b - a <= MERGE_GAP for a, b in pairwise(years))
for _, group in episodes.groupby("iso3", sort=True):
    ordered = [member_years[i] for i in group.sort_values(["opening_year", "case_id"]).index]
    for before, after in pairwise(ordered):
        if max(before) >= EPISODE_FIRST_YEAR and min(after) >= EPISODE_FIRST_YEAR:
            assert min(after) - max(before) > MERGE_GAP
print(
    "\nChecks passed: every catalogue case belongs to exactly one episode, the member lists agree with the episode investment, "
    "openings are chained only from the first episode year on and only within the merge gap, and the next start year is that of the next episode."
)

## Feasibility

An episode is feasible when its economy is in the panel, the outcome is observed in enough years before the opening and within the post-opening window, the donor pool is large enough, and no other opening of the economy lies inside the post-opening window. The openings that belong to the episode itself do not count as other openings. The donor pool of an episode holds the economies with a complete outcome series over the estimation window that have no opening of the full catalogue between the buffer period before the episode and the end of the window. The table of reasons counts every failed condition, so an episode that fails two conditions is counted twice.

In [ ]:
feasibility = check_feasibility(
    episodes,
    panel,
    outcome=OUTCOME,
    first_year=FIRST_YEAR,
    last_year=LAST_YEAR,
    post_horizon=POST_HORIZON,
    min_post=MIN_POST,
    catalogue=catalogue,
)
ctx.save_table(feasibility, "feasibility.csv")
assert episodes["case_id"].tolist() == feasibility["case_id"].tolist()

CONDITION_LABELS = {
    "n_pre": "too few pre-opening years with data",
    "n_post": "too few post-opening years with data",
    "n_donors": "too few donors",
    "economy": "economy not in the panel",
    "other_openings": "another opening of the same economy inside the post-opening window",
}


def failed_conditions(reason: str) -> list[str]:
    """Return the labels of the conditions named in a feasibility reason text.

    Parameters
    ----------
    reason : str
        Reason text of :func:`dtt.cases.check_feasibility`, with failed conditions separated by semicolons.

    Returns
    -------
    list of str
        One label per failed condition; empty for a feasible episode.
    """
    parts = [part.strip() for part in str(reason).split(";") if part.strip()]
    return [next((label for key, label in CONDITION_LABELS.items() if part.startswith(key)), part) for part in parts]


feasible = feasibility["feasible"].to_numpy(dtype=bool)
infeasible = feasibility.loc[~feasible]
print(f"Cases in the catalogue: {len(catalogue)}")
print(f"Episodes: {len(episodes)} in {episodes['iso3'].nunique()} economies")
print(f"Feasible episodes: {int(feasible.sum())} in {feasibility.loc[feasible, 'iso3'].nunique()} economies")
print(f"Infeasible episodes: {len(infeasible)}")
if len(infeasible):
    counts = pd.Series([label for text in infeasible["reason"] for label in failed_conditions(text)], dtype="str").value_counts()
    print("\nReasons for infeasibility (episodes failing each condition):")
    print(counts.rename("episodes").to_string())
    print("\nInfeasible episodes:")
    print(infeasible[["case_id", "n_pre", "n_post", "n_donors", "reason"]].to_string(index=False))
    unrecognised = sorted(set(counts.index) - set(CONDITION_LABELS.values()))
    assert not unrecognised, f"reasons without a label: {unrecognised}"
print(f"\nDonors per feasible episode: minimum {feasibility.loc[feasible, 'n_donors'].min()}, median {feasibility.loc[feasible, 'n_donors'].median():.0f}, maximum {feasibility.loc[feasible, 'n_donors'].max()}")

flag = episodes["panel_feasible"].astype("object").fillna("not given").to_numpy()
print("\nCatalogue flag panel_feasible against the rules above:")
print(pd.crosstab(flag, feasibility["feasible"].to_numpy(), rownames=["catalogue flag"], colnames=["feasible by the rules"]).to_string())

## Effects

The effect of an episode is estimated with the direct synthetic control: non-negative donor weights that sum to one are chosen to reproduce the outcome of the host economy in every pre-opening year with data. The effect in percentage points of GDP (`att_pp`) is the mean gap between the actual and the synthetic outcome over the post-opening window. The relative effect (`att_rel_pct`) is this mean gap as a percentage of the mean synthetic outcome of the window. The window starts in the opening year and ends after at most the post horizon, before the pandemic years and before the next episode of the same economy.

Placebo-in-space inference repeats the estimation for every donor, with the remaining donors as its pool. The standard error of an effect is the standard deviation of its placebo effects (`placebo_sd_pp` and `placebo_sd_rel_pct`). The rank p-value `p_gap_rank` is one plus the number of donors whose absolute placebo effect is at least as large as the absolute effect of the episode, divided by one plus the number of donors, so a tie counts against the episode. The ratio-based placebo p-value is not used, because the pre-opening fit is often exact and the ratio is then unstable.

The pre-opening fit is exact (`fit_exact`) when the gap between the actual and the synthetic outcome is within the tolerance of the settings cell in every pre-opening year. When an episode has fewer pre-opening years than donors, many different weight vectors then reproduce its pre-opening path, and the effect depends on which of them is chosen. The table reports the smallest and the largest effect over all non-negative weight vectors that sum to one and reproduce the pre-opening path (`att_pp_min` and `att_pp_max`); both equal `att_pp` when the fit is not exact. An episode meets the pre-opening fit criterion (`pre_fit_ok`) when its pre-opening root mean squared prediction error is at most the fit factor of the settings cell times the median of the placebo values, and the fit is not exact with fewer pre-opening years than donors. The relative effect is missing when the mean synthetic outcome of the window is not positive.

In [ ]:
started = time.perf_counter()
effects, placebos, results = estimate_all(
    panel,
    episodes,
    feasibility=feasibility,
    guard=ctx.guard,
    catalogue=catalogue,
    method="direct",
    seed=SEED,
    outcome=OUTCOME,
    first_year=FIRST_YEAR,
    last_year=LAST_YEAR,
    post_horizon=POST_HORIZON,
    pre_fit_factor=FIT_FACTOR,
)
print(f"Effects and placebo inference computed in {time.perf_counter() - started:.1f} seconds.")

shown = effects[
    [
        "case_id", "opening_year", "window_end", "n_pre", "n_donors", "att_pp", "att_pp_min", "att_pp_max", "att_rel_pct",
        "placebo_sd_pp", "p_gap_rank", "rmspe_pre", "fit_exact", "pre_fit_ok",
    ]
]
print(f"Effects estimated for {len(effects)} episodes in {effects['iso3'].nunique()} economies (direct weights, post horizon {POST_HORIZON}).\n")
print(shown.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print(
    f"\nEffect in percentage points of GDP: minimum {effects['att_pp'].min():.3f}, median {effects['att_pp'].median():.3f}, "
    f"maximum {effects['att_pp'].max():.3f}; {int((effects['att_pp'] > 0).sum())} of {len(effects)} are positive."
)
print(f"Episodes with p_gap_rank at most 0.10: {int((effects['p_gap_rank'] <= 0.10).sum())} of {len(effects)}.")
n_rel_missing = int(effects["att_rel_pct"].isna().sum())
print(f"The relative effect is missing for {n_rel_missing} of {len(effects)} episodes (mean synthetic outcome of the window not positive).")

fit_exact = effects["fit_exact"].to_numpy(dtype=bool)
non_unique_exact = fit_exact & (effects["n_pre"].to_numpy() < effects["n_donors"].to_numpy())
pre_fit_ok = effects["pre_fit_ok"].to_numpy(dtype=bool)
outside = ~pre_fit_ok
width = (effects["att_pp_max"] - effects["att_pp_min"]).to_numpy(dtype=float)
ratio = (width / effects["placebo_sd_pp"].to_numpy(dtype=float))[non_unique_exact & np.isfinite(width)]
print(f"\nFeasible episodes: {len(effects)}")
print(f"Episodes with an exact pre-opening fit: {int(fit_exact.sum())}; among them with fewer pre-opening years than donors: {int(non_unique_exact.sum())}.")
print(f"Outside the primary sample because the pre-opening fit is poor: {int((outside & ~non_unique_exact).sum())}")
print(f"Outside the primary sample because the fit is exact and not unique: {int((outside & non_unique_exact).sum())}")
if ratio.size:
    print(
        f"Median ratio of the width of the identified range of the effect to the placebo standard deviation, over these episodes: "
        f"{np.median(ratio):.2f} ({ratio.size} with a range)."
    )
else:
    print("No episode has an identified range to compare with the placebo standard deviation.")

# Checks against the definitions, recomputed from the stored paths and the placebo table.
assert len(effects) == int(feasible.sum())
assert effects["case_id"].is_unique and set(effects["case_id"]) == set(feasibility.loc[feasible, "case_id"])
assert np.isfinite(effects["att_pp"]).all() and (effects["placebo_sd_pp"] > 0).all()
assert effects["p_gap_rank"].between(0.0, 1.0).all() and (effects["p_gap_rank"] > 0.0).all()
assert (effects["post_horizon_used"] <= POST_HORIZON).all() and (effects["window_end"] <= LAST_YEAR).all()
assert (effects["n_post"] >= MIN_POST).all()
assert placebos.groupby("case_id").size().reindex(effects["case_id"]).to_numpy().tolist() == effects["n_donors"].tolist()

placebo_by_case = {case_id: group["placebo_att_pp"].to_numpy(dtype=float) for case_id, group in placebos.groupby("case_id")}
p_recomputed = []
for case_id, att in zip(effects["case_id"], effects["att_pp"], strict=True):
    draws = np.abs(placebo_by_case[case_id])
    draws = draws[np.isfinite(draws)]
    p_recomputed.append((1.0 + np.count_nonzero(draws >= abs(att))) / (1.0 + draws.size))
assert np.allclose(p_recomputed, effects["p_gap_rank"].to_numpy(dtype=float), rtol=0.0, atol=1e-12)
sd_recomputed = placebos.groupby("case_id")["placebo_att_pp"].std(ddof=1).reindex(effects["case_id"]).to_numpy()
assert np.allclose(sd_recomputed, effects["placebo_sd_pp"].to_numpy(dtype=float), rtol=1e-9, atol=0.0)

placebo_median = placebos.groupby("case_id")["placebo_rmspe_pre"].median().reindex(effects["case_id"]).to_numpy()
margin = effects["rmspe_pre"].to_numpy() - FIT_FACTOR * placebo_median
expected_ok = (margin <= 0.0) & ~non_unique_exact
decidable = (np.abs(margin) > 1e-9) | non_unique_exact
assert (expected_ok == pre_fit_ok)[decidable].all(), "pre_fit_ok differs from the RMSPE rule combined with the exact-fit rule"
assert not (pre_fit_ok & non_unique_exact).any()


def window_means(res) -> tuple[float, float]:
    """Mean gap and mean synthetic outcome over the post-opening window, recomputed from the stored paths.

    Parameters
    ----------
    res : dtt.effects.CaseEffect
        Result of one episode.

    Returns
    -------
    tuple of float
        Mean gap and mean synthetic outcome in the years of the window with an observed outcome.
    """
    in_window = (res.years >= res.opening_year) & (res.years <= res.window_end) & np.isfinite(res.actual)
    return float(np.mean(res.gap[in_window])), float(np.mean(res.synthetic[in_window]))


def largest_pre_gap(res) -> float:
    """Largest absolute gap in the pre-opening years with an observed outcome, from the stored paths.

    Parameters
    ----------
    res : dtt.effects.CaseEffect
        Result of one episode.

    Returns
    -------
    float
        The largest absolute gap in outcome units.
    """
    gap = res.gap[res.years < res.opening_year]
    return float(np.max(np.abs(gap[np.isfinite(gap)])))


means = np.array([window_means(results[c]) for c in effects["case_id"]])
level_positive = means[:, 1] > 0.0
assert np.allclose(means[:, 0], effects["att_pp"].to_numpy(dtype=float), rtol=0.0, atol=1e-9)
assert (effects["att_rel_pct"].isna().to_numpy() == ~level_positive).all(), "att_rel_pct is missing outside the documented case"
assert np.allclose(
    100.0 * means[level_positive, 0] / means[level_positive, 1], effects.loc[level_positive, "att_rel_pct"].to_numpy(dtype=float), rtol=1e-9
)

pre_gap = np.array([largest_pre_gap(results[c]) for c in effects["case_id"]])
clear = np.abs(pre_gap - EXACT_FIT_TOLERANCE) > 1e-12
assert (fit_exact == (pre_gap <= EXACT_FIT_TOLERANCE))[clear].all()
assert np.allclose(effects.loc[~fit_exact, ["att_pp_min", "att_pp_max"]].to_numpy(dtype=float), effects.loc[~fit_exact, ["att_pp"]].to_numpy(dtype=float))
has_range = effects[["att_pp_min", "att_pp_max"]].notna().all(axis=1).to_numpy()
assert (effects.loc[has_range, "att_pp_min"] <= effects.loc[has_range, "att_pp_max"] + 1e-9).all()
nonnegative_weights = np.array([results[c].weights.min() >= -1e-12 for c in effects["case_id"]])
contains = ((effects["att_pp_min"] - 1e-9 <= effects["att_pp"]) & (effects["att_pp"] <= effects["att_pp_max"] + 1e-9)).to_numpy()
assert contains[has_range & nonnegative_weights].all()
print(
    "\nChecks passed: one effect per feasible episode, finite effects, rank p-values and placebo standard deviations equal to their definitions, "
    "windows within the limits, the fit flag equal to the RMSPE rule combined with the exact-fit rule, the relative effect missing only where the "
    "synthetic level is not positive, and the identified range consistent with the exact-fit flag."
)

## Features

The features describe the host economy before the opening and use only panel years before the opening year. The level features are means over the years of the feature window; the growth feature is the slope of log receipts over the trailing years before the opening, with the window length fixed in `dtt.cases`; the investment feature is the investment of the episode as a percentage of GDP; for an episode with several openings it is the sum, over the openings that take place no later than the end of the post-opening window, of each investment divided by the GDP of the year before that opening; the resort feature is one for an integrated or destination resort. The reference table gives the same features for every economy in every year of the reference period, with the levels taken from that single year. A missing episode feature is filled with the median of the reference rows from years before the opening year of the episode. If the reference holds no such value, the median of the same feature over the other episodes that open no later is used. A feature is dropped when its share of missing values exceeds the limit or when a missing value cannot be filled in this way. In the imputation report, `fill_value` is the mean of the values filled in and `fill_source` says whether they came from the reference table, from the other episodes or from both. Only the estimated episodes enter the feature tables, so the missing shares refer to the analysis sample.

In [ ]:
reference_frames = []
for k, iso3 in enumerate(economies):
    reference_frames.append(economy_feature_cloud(panel, iso3, REFERENCE_YEARS))
    if ctx.guard and (k + 1) % 25 == 0:
        ctx.guard()
reference = pd.concat(reference_frames, ignore_index=True)
ctx.save_table(reference, "reference_features.csv")

estimated_episodes = episodes.set_index("case_id").loc[effects["case_id"]].reset_index()
features = case_features(estimated_episodes, panel, window=FEATURE_WINDOW, last_year=LAST_YEAR, post_horizon=POST_HORIZON)
ctx.save_table(features, "case_features.csv")
imputed, imputation_report = impute_features(features, reference=reference, max_missing_share=MAX_MISSING_SHARE)
ctx.save_table(imputed, "case_features_imputed.csv")
ctx.save_table(imputation_report, "imputation_report.csv")

print(f"Reference table: {len(reference)} economy-years for {reference['iso3'].nunique()} economies, years {reference['year'].min()} to {reference['year'].max()}.")
print(f"Episode features: {len(features)} episodes, {len(FEATURE_COLUMNS)} features.\n")
print("Imputation report:")
print(imputation_report.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
dropped = imputation_report.loc[imputation_report["dropped"], "feature"].tolist()
kept = [c for c in FEATURE_COLUMNS if c in imputed.columns]
n_filled = int(imputation_report.loc[~imputation_report["dropped"], "n_missing"].sum())
print(f"\nFeatures dropped: {', '.join(dropped) if dropped else 'none'}")
for row in imputation_report.loc[imputation_report["dropped"]].itertuples():
    print(f"  {row.feature}: {row.reason}")
print(f"Features kept: {len(kept)}; values imputed: {n_filled}")
sources = imputation_report.loc[~imputation_report["dropped"] & (imputation_report["n_missing"] > 0), "fill_source"].value_counts()
if len(sources):
    print("\nFeatures with imputed values, by source of the fill values:")
    print(sources.rename("features").rename_axis("fill_source").to_string())

assert features["case_id"].tolist() == effects["case_id"].tolist() == imputed["case_id"].tolist()
assert set(reference["iso3"]) == set(economies) and set(reference["year"]) == set(REFERENCE_YEARS)
assert list(reference.columns) == ["iso3", "year", *FEATURE_COLUMNS]
assert list(features.columns) == ["case_id", "iso3", "opening_year", *FEATURE_COLUMNS]
assert list(imputed.columns) == ["case_id", "iso3", "opening_year", *kept]
assert np.isfinite(imputed[kept].to_numpy(dtype=float)).all()
assert set(kept) | set(dropped) == set(FEATURE_COLUMNS) and not set(kept) & set(dropped)
assert imputation_report["feature"].tolist() == list(FEATURE_COLUMNS)

# The imputation follows the documented rule, re-implemented here for every feature and every missing cell.
openings = features["opening_year"].to_numpy(dtype=float)
report_by_feature = imputation_report.set_index("feature")
n_checked = 0
for name in FEATURE_COLUMNS:
    column = features[name].to_numpy(dtype=float)
    missing = ~np.isfinite(column)
    row = report_by_feature.loc[name]
    assert int(row["n_missing"]) == int(missing.sum()) and np.isclose(row["share_missing"], missing.mean())
    fills, labels, unfilled = [], [], 0
    for i in np.flatnonzero(missing):
        pool = reference.loc[reference["year"] < openings[i], name].to_numpy(dtype=float)
        pool, label = pool[np.isfinite(pool)], "reference"
        if pool.size == 0:
            pool, label = column[~missing & (openings <= openings[i])], "own"
        if pool.size == 0:
            unfilled += 1
        else:
            fills.append(float(np.median(pool)))
            labels.append(label)
    should_drop = bool(missing.mean() > MAX_MISSING_SHARE or unfilled > 0)
    assert bool(row["dropped"]) == should_drop, f"{name}: dropped flag differs from the rule"
    assert (name not in imputed.columns) == should_drop
    assert bool(str(row["reason"]).strip()) == should_drop, f"{name}: reason text does not match the dropped flag"
    if should_drop or not missing.any():
        assert row["fill_source"] == "none" and not np.isfinite(row["fill_value"]), name
        continue
    assert row["fill_source"] == (labels[0] if len(set(labels)) == 1 else "mixed"), f"{name}: fill source differs from the rule"
    assert np.allclose(imputed.loc[missing, name].to_numpy(dtype=float), fills, rtol=1e-12, atol=0.0), name
    assert np.isclose(row["fill_value"], np.mean(fills), rtol=1e-12, atol=0.0), name
    n_checked += len(fills)
assert n_checked == n_filled

# The investment feature counts only the openings inside the post-opening window, each over the GDP of the year before its own opening.
gdp = panel.set_index(["iso3", "year"])["gdp_usd"]
expected_capex = []
n_members = n_counted = 0
for row in estimated_episodes.itertuples():
    years = [int(v) for v in str(row.member_opening_years).split(";")]
    amounts = [float(v) if v.strip() else np.nan for v in str(row.member_investment_usd_bn).split(";")]
    window_end = results[row.case_id].window_end
    inside = [(year, amount) for year, amount in zip(years, amounts, strict=True) if year <= max(window_end, row.opening_year)]
    n_members += len(years)
    n_counted += len(inside)
    shares = [100.0 * amount * 1e9 / gdp.get((row.iso3, year - 1), np.nan) for year, amount in inside if np.isfinite(amount)]
    expected_capex.append(float(sum(shares)) if shares and np.isfinite(shares).all() else np.nan)
assert np.allclose(features["capex_pct_gdp"].to_numpy(dtype=float), expected_capex, rtol=1e-9, atol=0.0, equal_nan=True)
print(
    f"\nOpenings of the estimated episodes: {n_members}. Inside the post-opening window of their episode, and so in the investment feature: "
    f"{n_counted}. Later openings that do not enter it: {n_members - n_counted}."
)
print(
    "\nChecks passed: the imputed feature table has the rows of the effects table and no missing values, "
    f"the dropped features, the fill sources and all {n_checked} imputed values follow the documented rule, and the investment feature follows its definition."
)

## Sample flags and the effects table

The primary analysis sample is the set of feasible episodes that meet the pre-opening fit criterion; the flag `sample_primary` marks it. Two kinds of feasible episodes are set aside: those whose pre-opening fit is poor compared with the fits of their placebos, and those whose pre-opening path is reproduced exactly while the donors outnumber the pre-opening years, because their weights are not unique and their effect is identified only as a range. The printed counts say how many episodes fall in each group. The flag `parks_only` marks episodes whose category is a theme park or a multi-park resort, and the flag `first_episode` marks the earliest feasible episode of each economy. Economies, not episodes, are the independent units, so the numbers of economies are reported next to the numbers of episodes. The sensitivity samples of notebook 03 are the primary sample restricted by `parks_only`, the primary sample restricted by `first_episode`, and all feasible episodes.

In [ ]:
meta = episodes.set_index("case_id")
case_effects = effects.copy()
case_effects["sample_primary"] = case_effects["pre_fit_ok"].astype(bool)
case_effects["category"] = case_effects["case_id"].map(meta["category"])
case_effects["parks_only"] = case_effects["category"].isin(PARK_CATEGORIES)
case_effects["first_episode"] = case_effects["opening_year"] == case_effects.groupby("iso3")["opening_year"].transform("min")
case_effects["capex_pct_gdp"] = case_effects["case_id"].map(features.set_index("case_id")["capex_pct_gdp"])
case_effects["n_openings"] = case_effects["case_id"].map(meta["n_openings"]).astype(int)

primary = case_effects["sample_primary"].to_numpy()
samples = {
    "All feasible episodes": np.ones(len(case_effects), dtype=bool),
    "Primary sample": primary,
    "Parks only, within the primary sample": primary & case_effects["parks_only"].to_numpy(),
    "First episode of each economy, within the primary sample": primary & case_effects["first_episode"].to_numpy(),
}
sample_table = pd.DataFrame(
    [
        {"sample": name, "episodes": int(mask.sum()), "economies": int(case_effects.loc[mask, "iso3"].nunique())}
        for name, mask in samples.items()
    ]
)
print(sample_table.to_string(index=False))
n_primary, n_primary_economies = int(primary.sum()), case_effects.loc[primary, "iso3"].nunique()
print(f"\nThe primary sample has {n_primary} episodes in {n_primary_economies} economies.")
set_aside_exact = ~primary & (case_effects["fit_exact"] & (case_effects["n_pre"] < case_effects["n_donors"])).to_numpy()
set_aside_poor = ~primary & ~set_aside_exact
print(
    f"Feasible episodes outside the primary sample: {int((~primary).sum())} of {len(case_effects)}; {int(set_aside_poor.sum())} because their "
    f"pre-opening fit is poor and {int(set_aside_exact.sum())} because the fit is exact and not unique."
)
if set_aside_exact.any():
    print("Set aside because the fit is exact and not unique: " + ", ".join(case_effects.loc[set_aside_exact, "case_id"]))
if set_aside_poor.any():
    print("Set aside because the pre-opening fit is poor: " + ", ".join(case_effects.loc[set_aside_poor, "case_id"]))
multi_economy = case_effects.groupby("iso3").size()
print(f"Economies with more than one feasible episode: {int((multi_economy > 1).sum())}.")
print(f"Episodes with a known investment cost: {int(case_effects['capex_pct_gdp'].notna().sum())} of {len(case_effects)}.")

EFFECT_FILE_COLUMNS = [*effects.columns, "sample_primary", "first_episode", "parks_only", "category", "capex_pct_gdp", "n_openings"]
case_effects = case_effects[EFFECT_FILE_COLUMNS]
ctx.save_table(case_effects, "case_effects.csv")
ctx.save_table(placebos, "case_placebos.csv")

assert case_effects["case_id"].is_unique and len(case_effects) == len(effects)
assert case_effects.groupby("iso3")["first_episode"].sum().eq(1).all()
assert (case_effects["sample_primary"] == case_effects["pre_fit_ok"]).all()
assert int(set_aside_poor.sum() + set_aside_exact.sum() + primary.sum()) == len(case_effects)
assert list(case_effects.columns[: len(effects.columns)]) == list(effects.columns)
print("\nChecks passed: each economy has exactly one first episode, the flags are consistent, and every feasible episode is either in the primary sample or set aside for one named reason.")

## Robustness variants

The variants check whether an effect depends on the weighting method, on the length of the window or on the estimator. The ridge variants augment the direct weights by a ridge regression of the remaining pre-opening imbalance (Ben-Michael, Feller and Rothstein, 2021); the weights still sum to one but may be negative, and the pre-opening fit can only improve. The penalty is added to the cross-product matrix of the donors' pre-opening outcomes after centring over donors, so its natural unit is the average diagonal element of that matrix: the sum over donors of the squared deviation from the cross-donor mean, averaged over the pre-opening years. The penalties are fixed fractions of the median of that unit across the estimated episodes; the fractions are set in the settings cell and the resulting penalties are printed below. The horizon variants shorten and lengthen the post-opening window. The difference-in-differences variant compares the change of the episode with the change of the equally weighted donors. The comparison table also gives the largest difference within the primary sample, because the direct effect of an episode with an exact fit and more donors than pre-opening years is only one point of its identified range.

In [ ]:
wide = outcome_matrix(panel, OUTCOME, FIRST_YEAR, LAST_YEAR)
scales = []
for k, (_, row) in enumerate(estimated_episodes.iterrows()):
    if ctx.guard and k % 10 == 0:
        ctx.guard()
    pool = donor_pool(panel, row, episodes, outcome=OUTCOME, first_year=FIRST_YEAR, last_year=LAST_YEAR, catalogue=catalogue)
    observed_pre = (wide.index < row["opening_year"]) & wide[row["iso3"]].notna().to_numpy()
    donor_pre = wide.loc[observed_pre, pool].to_numpy(dtype=float)
    centred = donor_pre - donor_pre.mean(axis=1, keepdims=True)
    scales.append(float(np.mean(np.sum(centred**2, axis=1))))
ridge_scale = float(np.median(scales))
ridge_penalties = {name: fraction * ridge_scale for name, fraction in RIDGE_FRACTIONS.items()}
print(
    f"Scale of the ridge penalty (median over {len(scales)} episodes of the average sum of squared deviations from the cross-donor mean): "
    f"{ridge_scale:.2f}; range {min(scales):.2f} to {max(scales):.2f}"
)
for name, penalty in ridge_penalties.items():
    print(f"{name}: {RIDGE_FRACTIONS[name]} times the scale = {penalty:.3f}")

common = dict(
    guard=ctx.guard,
    catalogue=catalogue,
    seed=SEED,
    outcome=OUTCOME,
    first_year=FIRST_YEAR,
    last_year=LAST_YEAR,
    pre_fit_factor=FIT_FACTOR,
)
started = time.perf_counter()
variant_effects = {"direct": effects}
for name, penalty in ridge_penalties.items():
    if ctx.guard:
        ctx.guard()
    variant_effects[name], _, _ = estimate_all(
        panel, episodes, feasibility=feasibility, method="ridge", ridge=penalty, post_horizon=POST_HORIZON, **common
    )
for horizon in (SHORT_HORIZON, LONG_HORIZON):
    if ctx.guard:
        ctx.guard()
    feasibility_h = check_feasibility(
        episodes, panel, outcome=OUTCOME, first_year=FIRST_YEAR, last_year=LAST_YEAR,
        post_horizon=horizon, min_post=MIN_POST, catalogue=catalogue,
    )
    variant_effects[f"horizon{horizon}"], _, _ = estimate_all(
        panel, episodes, feasibility=feasibility_h, method="direct", post_horizon=horizon, **common
    )

frames = [
    table[["case_id", "att_pp", "att_rel_pct", "rmspe_pre"]].assign(variant=name) for name, table in variant_effects.items()
]
frames.append(
    effects[["case_id", "did_att_pp"]]
    .rename(columns={"did_att_pp": "att_pp"})
    .assign(variant="did", att_rel_pct=np.nan, rmspe_pre=np.nan)
)
robustness = pd.concat(frames, ignore_index=True)[["case_id", "variant", "att_pp", "att_rel_pct", "rmspe_pre"]]
ctx.save_table(robustness, "case_effects_robustness.csv")
print(f"Robustness variants computed in {time.perf_counter() - started:.1f} seconds.")

att = robustness.pivot(index="case_id", columns="variant", values="att_pp").reindex(effects["case_id"])
primary_ids = set(effects.loc[effects["pre_fit_ok"], "case_id"])
rows = []
for name in att.columns:
    if name == "direct":
        continue
    both = att[["direct", name]].dropna()
    gap = both[name] - both["direct"]
    rows.append(
        {
            "variant": name,
            "episodes": len(both),
            "correlation with direct": float(np.corrcoef(both["direct"], both[name])[0, 1]) if len(both) > 2 else np.nan,
            "mean difference": float(gap.mean()),
            "largest absolute difference": float(gap.abs().max()),
            "largest absolute difference, primary sample": float(gap[both.index.isin(primary_ids)].abs().max()) if both.index.isin(primary_ids).any() else np.nan,
            "same sign as direct": int((np.sign(both[name]) == np.sign(both["direct"])).sum()),
        }
    )
print("\nEffects in percentage points of GDP of each variant compared with the direct estimator:")
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.3f}"))

expected_variants = {"direct", *ridge_penalties, f"horizon{SHORT_HORIZON}", f"horizon{LONG_HORIZON}", "did"}
assert set(robustness["variant"]) == expected_variants == {"direct", "ridge_low_penalty", "ridge_high_penalty", "horizon3", "horizon8", "did"}
assert set(robustness["case_id"]) <= set(effects["case_id"])
assert not robustness.duplicated(["case_id", "variant"]).any()
assert np.isfinite(robustness["att_pp"]).all()
direct_pre = effects.set_index("case_id")["rmspe_pre"]
for name in ridge_penalties:
    ridge_pre = variant_effects[name].set_index("case_id")["rmspe_pre"]
    assert (ridge_pre <= direct_pre.reindex(ridge_pre.index) + 1e-8).all(), name
assert (variant_effects[f"horizon{SHORT_HORIZON}"]["post_horizon_used"] <= SHORT_HORIZON).all()
assert (variant_effects[f"horizon{LONG_HORIZON}"]["post_horizon_used"] <= LONG_HORIZON).all()
print("\nChecks passed: each variant has at most one row per estimated episode, effects are finite, and a ridge pre-opening fit is no worse than the direct fit.")

## Figure: effects with the placebo range

Each row is an episode, sorted by its effect with the largest effect at the top. The grey bar spans the central range of the placebo effects of the donors of that episode, with the percentile limits given in the legend; an effect inside the bar cannot be distinguished from the effects of untreated donors. Filled markers are episodes of the primary sample and open markers are episodes outside it. A thin black line through the marker of an episode with an exact pre-opening fit spans its identified range, the smallest to the largest effect that any admissible weighting gives (`att_pp_min` to `att_pp_max`); the line is short where the weights are unique.

In [ ]:
def legend_below(ax: plt.Axes, **kwargs: object) -> Legend:
    """Place the legend of an axes under its x axis label, whatever the size of the figure.

    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Axes with a finished x axis label and labelled artists.
    **kwargs
        Passed to ``Axes.legend``.

    Returns
    -------
    matplotlib.legend.Legend
        The legend.
    """
    fig = ax.figure
    fig.canvas.draw()
    label_box = ax.xaxis.label.get_window_extent(fig.canvas.get_renderer())
    y_label = ax.transAxes.inverted().transform((0.0, label_box.y0))[1]
    return ax.legend(loc="upper center", bbox_to_anchor=(0.5, y_label - 0.03), **kwargs)


LOW_Q, HIGH_Q = PLACEBO_INTERVAL
interval_name = f"{100 * LOW_Q:.0f}th to {100 * HIGH_Q:.0f}th percentile"
interval = placebos.groupby("case_id")["placebo_att_pp"].quantile([LOW_Q, HIGH_Q]).unstack().reindex(case_effects["case_id"])
forest = case_effects.assign(low=interval[LOW_Q].to_numpy(), high=interval[HIGH_Q].to_numpy()).sort_values("att_pp").reset_index(drop=True)
y = np.arange(len(forest))
labels = [
    f"{r.iso3} {r.opening_year}" + (f" ({r.n_openings} openings)" if r.n_openings > 1 else "") for r in forest.itertuples()
]
inside = int(((forest["att_pp"] >= forest["low"]) & (forest["att_pp"] <= forest["high"])).sum())

fig, ax = plt.subplots(figsize=(8.0, max(4.0, 0.27 * len(forest) + 1.8)))
ax.hlines(y, forest["low"], forest["high"], color="0.72", linewidth=6, label=f"Placebo effects, {interval_name}")
has_range = (forest["fit_exact"] & forest["att_pp_min"].notna() & forest["att_pp_max"].notna()).to_numpy()
if has_range.any():
    ax.hlines(y[has_range], forest.loc[has_range, "att_pp_min"], forest.loc[has_range, "att_pp_max"], color="0.0", linewidth=1.3, label="Identified range of the effect (exact pre-opening fit)")
    ax.plot(
        np.r_[forest.loc[has_range, "att_pp_min"], forest.loc[has_range, "att_pp_max"]], np.r_[y[has_range], y[has_range]],
        linestyle="none", marker="|", color="0.0", markersize=7,
    )
keep = forest["sample_primary"].to_numpy()
ax.plot(forest.loc[keep, "att_pp"], y[keep], linestyle="none", marker="o", color="0.0", markersize=5.5, label="Effect, primary sample")
ax.plot(
    forest.loc[~keep, "att_pp"], y[~keep], linestyle="none", marker="o", markerfacecolor="white", markeredgecolor="0.0",
    markersize=5.5, label="Effect, outside the primary sample" if (~keep).any() else None,
)
ax.axvline(0.0, color="0.3", linewidth=0.9)
ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=8)
ax.set_ylim(-0.8, len(forest) - 0.2)
ax.set_xlabel("Effect on receipts: mean gap over the post-opening window (percentage points of GDP)")
ax.set_ylabel("Episode (economy and opening year)")
ax.set_title("Effect of each episode and the range of its placebo effects")
legend_below(ax, fontsize=8, ncol=1)
ctx.save_figure(fig, "case_effects_forest.png")
print(f"{inside} of {len(forest)} effects lie inside the {interval_name} range of their own placebo effects.")
print(f"Identified ranges drawn: {int(has_range.sum())}; exact fits without a range to draw: {int((forest['fit_exact'] & ~has_range).sum())}.")

## Figure: actual, synthetic and placebo gap paths

The figure shows the episodes of the primary sample with the most donors, up to the number named in the settings cell. For each episode the left panel compares the observed outcome with the synthetic control, and the right panel shows the gap between the two (black) together with the gaps of the placebo fits of all donors (grey). The shaded years are the post-opening window. The vertical axis of the gap panel is limited, so a few placebo paths with a poor pre-opening fit may leave the panel.

In [ ]:
chosen = (
    case_effects.loc[case_effects["sample_primary"]]
    .sort_values(["n_donors", "case_id"], ascending=[False, True])["case_id"]
    .head(MAX_PATH_EPISODES)
    .tolist()
)
n_rows = max(1, int(np.ceil(len(chosen) / 2)))
fig, axes = plt.subplots(n_rows, 4, figsize=(15.5, 3.4 * n_rows + 0.9), squeeze=False)
for k, case_id in enumerate(chosen):
    res = results[case_id]
    ax_level, ax_gap = axes[k // 2, 2 * (k % 2)], axes[k // 2, 2 * (k % 2) + 1]
    for ax in (ax_level, ax_gap):
        ax.axvspan(res.opening_year - 0.5, res.window_end + 0.5, facecolor="0.9", edgecolor="none")
        ax.axvline(res.opening_year - 0.5, color="0.35", linestyle=":", linewidth=1.0)
        ax.set_xlim(res.years.min() - 0.5, res.years.max() + 0.5)
        ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=6))
        ax.set_xlabel("Year")
    ax_level.plot(res.years, res.actual, color="0.0", linestyle="-", marker="o", markersize=2.5, linewidth=1.3)
    ax_level.plot(res.years, res.synthetic, color="0.45", linestyle="--", linewidth=1.5)
    ax_level.set_ylabel("Receipts (percent of GDP)")
    ax_level.set_title(f"{res.iso3}, opening {res.opening_year}, {res.n_donors} donors", fontsize=9)
    ax_gap.plot(res.years, res.placebo_gaps, color="0.72", linestyle="-", linewidth=0.5)
    ax_gap.plot(res.years, res.gap, color="0.0", linestyle="-", marker="o", markersize=2.5, linewidth=1.6)
    ax_gap.axhline(0.0, color="0.3", linewidth=0.8)
    limit = max(1.2 * float(np.nanmax(np.abs(res.gap))), 2.5 * float(np.std(res.placebo_att_pp, ddof=1)))
    ax_gap.set_ylim(-limit, limit)
    ax_gap.set_ylabel("Gap (percentage points of GDP)")
    row = case_effects.set_index("case_id").loc[case_id]
    ax_gap.set_title(f"Effect {row['att_pp']:.2f} pp, rank p-value {row['p_gap_rank']:.3f}", fontsize=9)
for k in range(len(chosen), 2 * n_rows):
    axes[k // 2, 2 * (k % 2)].set_visible(False)
    axes[k // 2, 2 * (k % 2) + 1].set_visible(False)
fig.legend(
    handles=[
        Line2D([0], [0], color="0.0", linestyle="-", marker="o", markersize=3, label="Actual outcome and gap of the episode"),
        Line2D([0], [0], color="0.45", linestyle="--", label="Synthetic control"),
        Line2D([0], [0], color="0.72", linestyle="-", label="Placebo gaps of the donors"),
        Patch(facecolor="0.9", edgecolor="0.6", label="Post-opening window"),
    ],
    loc="lower center",
    ncol=4,
    fontsize=9,
    bbox_to_anchor=(0.5, -0.01),
)
fig.suptitle("Actual, synthetic and placebo gap paths for the episodes with the most donors", y=0.995)
fig.tight_layout(rect=(0, 0.04, 1, 0.97))
ctx.save_figure(fig, "case_gap_paths.png")
print("Episodes shown: " + ", ".join(chosen))

## Figure: effect against investment

Each point is an episode with a known investment cost. The horizontal axis is the investment as a percentage of GDP in the year before the opening. Filled markers are episodes of the primary sample and open markers are episodes outside it. The straight line is the ordinary least squares fit to all plotted points and is drawn only when enough episodes have a cost, as set in the settings cell. It describes the association in the sample and is not an estimate of the effect of investment.

In [ ]:
def place_labels(
    ax: plt.Axes,
    xs: np.ndarray,
    ys: np.ndarray,
    texts: list[str],
    fontsize: float = 7.0,
    lines: list[tuple[np.ndarray, np.ndarray]] | None = None,
) -> None:
    """Write a short label next to each point, at the free position closest to the point.

    A position is free when the label lies inside the axes and overlaps no earlier label, no other marker and no
    drawn line; when no position is free, the one with the fewest overlaps is used. A label that is not next to its
    point is joined to the point by a thin line.

    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Axes that hold the points; the axis limits must already be final.
    xs, ys : numpy.ndarray
        Point coordinates in data units.
    texts : list of str
        One label per point.
    fontsize : float, default 7.0
        Font size of the labels in points.
    lines : list of tuple of numpy.ndarray, optional
        Paths drawn in the axes, each as a pair of x and y coordinates in data units; labels avoid them.
    """
    fig = ax.figure
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    frame = ax.get_window_extent(renderer)
    centres = ax.transData.transform(np.column_stack([xs, ys]))
    traced = []
    for line_x, line_y in lines or []:
        along = np.linspace(0.0, 1.0, len(line_x))
        steps = np.linspace(0.0, 1.0, 400)
        traced.append(ax.transData.transform(np.column_stack([np.interp(steps, along, line_x), np.interp(steps, along, line_y)])))
    path_points = np.vstack(traced) if traced else np.empty((0, 2))
    half = 4.5 * fig.dpi / 72.0
    markers = [Bbox.from_extents(cx - half, cy - half, cx + half, cy + half) for cx, cy in centres]
    offsets = [
        (6, 3), (6, -10), (-6, 3), (-6, -10), (0, 8), (0, -14), (13, 0), (-13, 0), (13, 9), (-13, 9), (13, -15), (-13, -15),
        (20, 4), (-20, 4), (20, -12), (-20, -12), (0, 16), (0, -22), (24, 14), (-24, 14), (24, -20), (-24, -20),
    ]
    taken: list[Bbox] = []
    for i, (x, y, text) in enumerate(zip(xs, ys, texts, strict=True)):
        others = [m for j, m in enumerate(markers) if j != i]
        best = None
        for dx, dy in offsets:
            note = ax.annotate(
                text, (x, y), xytext=(dx, dy), textcoords="offset points", fontsize=fontsize,
                ha="center" if dx == 0 else ("left" if dx > 0 else "right"), va="bottom",
            )
            box = note.get_window_extent(renderer)
            note.remove()
            inside_frame = frame.x0 <= box.x0 and box.x1 <= frame.x1 and frame.y0 <= box.y0 and box.y1 <= frame.y1
            on_line = bool(
                path_points.size
                and np.any((path_points[:, 0] >= box.x0) & (path_points[:, 0] <= box.x1) & (path_points[:, 1] >= box.y0) & (path_points[:, 1] <= box.y1))
            )
            cost = (0 if inside_frame else 10) + int(on_line) + sum(box.overlaps(other) for other in taken + others)
            if best is None or cost < best[0]:
                best = (cost, dx, dy, box)
            if cost == 0:
                break
        _, dx, dy, box = best
        far = max(abs(dx), abs(dy)) >= 16
        ax.annotate(
            text, (x, y), xytext=(dx, dy), textcoords="offset points", fontsize=fontsize,
            ha="center" if dx == 0 else ("left" if dx > 0 else "right"), va="bottom",
            arrowprops={"arrowstyle": "-", "color": "0.5", "linewidth": 0.6, "shrinkA": 0, "shrinkB": 3} if far else None,
        )
        taken.append(box)


with_cost = case_effects.dropna(subset=["capex_pct_gdp", "att_pp"]).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(8.2, 6.0))
keep = with_cost["sample_primary"].to_numpy()
ax.plot(with_cost.loc[keep, "capex_pct_gdp"], with_cost.loc[keep, "att_pp"], linestyle="none", marker="o", color="0.0", markersize=6, label="Primary sample")
ax.plot(
    with_cost.loc[~keep, "capex_pct_gdp"], with_cost.loc[~keep, "att_pp"], linestyle="none", marker="o",
    markerfacecolor="white", markeredgecolor="0.0", markersize=6, label="Outside the primary sample" if (~keep).any() else None,
)
slope = intercept = np.nan
drawn_lines = []
if len(with_cost) >= MIN_POINTS_FOR_LINE:
    slope, intercept = np.polyfit(with_cost["capex_pct_gdp"], with_cost["att_pp"], 1)
    grid_x = np.linspace(0.0, with_cost["capex_pct_gdp"].max() * 1.05, 50)
    ax.plot(grid_x, intercept + slope * grid_x, color="0.35", linestyle="--", linewidth=1.5, label=f"Least squares line, slope {slope:.2f}")
    drawn_lines.append((grid_x, intercept + slope * grid_x))
ax.set_xlim(left=0.0, right=with_cost["capex_pct_gdp"].max() * 1.12)
span = float(with_cost["att_pp"].max() - with_cost["att_pp"].min())
ax.set_ylim(with_cost["att_pp"].min() - 0.08 * span, with_cost["att_pp"].max() + 0.12 * span)
ax.set_xlabel("Investment in the year before the opening (percent of GDP)")
ax.set_ylabel("Effect on receipts (percentage points of GDP)")
ax.set_title("Effect against investment")
ax.axhline(0.0, color="0.5", linewidth=0.8)
legend_below(ax, ncol=3, fontsize=8)
place_labels(ax, with_cost["capex_pct_gdp"].to_numpy(), with_cost["att_pp"].to_numpy(), with_cost["iso3"].tolist(), lines=drawn_lines)
ctx.save_figure(fig, "case_effect_vs_capex.png")
print(f"{len(with_cost)} of {len(case_effects)} episodes have a known investment cost and are plotted.")
if len(with_cost) >= MIN_POINTS_FOR_LINE:
    r = float(np.corrcoef(with_cost["capex_pct_gdp"], with_cost["att_pp"])[0, 1])
    print(f"Fitted line: effect = {intercept:.3f} + {slope:.3f} x investment share; correlation {r:.3f}.")
else:
    print(f"Fewer than {MIN_POINTS_FOR_LINE} episodes have a cost, so no line is drawn.")

## Figure: the estimators against each other

Each panel compares the effects of the same episodes under a pair of estimators chosen from the direct synthetic control, the ridge-augmented synthetic control with each of its penalties and the difference-in-differences estimator. The dashed line is the 45-degree line, on which the two estimators agree.

In [ ]:
pairs = [
    ("direct", ["ridge_low_penalty", "ridge_high_penalty"], "Direct against ridge"),
    ("direct", ["did"], "Direct against difference-in-differences"),
    ("ridge_low_penalty", ["did"], "Ridge (low penalty) against difference-in-differences"),
]
names = {
    "direct": "direct synthetic control",
    "ridge_low_penalty": "ridge, low penalty",
    "ridge_high_penalty": "ridge, high penalty",
    "did": "difference-in-differences",
}
markers = {"ridge_low_penalty": "o", "ridge_high_penalty": "s", "did": "^"}
fig, axes = plt.subplots(1, 3, figsize=(15.0, 5.4))
for ax, (x_name, y_names, title) in zip(axes, pairs, strict=True):
    values = [att[x_name].to_numpy()] + [att[n].to_numpy() for n in y_names]
    lo = float(np.nanmin(np.concatenate(values)))
    hi = float(np.nanmax(np.concatenate(values)))
    pad = 0.06 * (hi - lo if hi > lo else 1.0)
    ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], color="0.35", linestyle="--", linewidth=1.2, label="45-degree line")
    for y_name, fill in zip(y_names, ("0.0", "none"), strict=False):
        ax.plot(
            att[x_name], att[y_name], linestyle="none", marker=markers[y_name], markersize=6,
            markerfacecolor=fill, markeredgecolor="0.0", label=names[y_name],
        )
    ax.set_xlim(lo - pad, hi + pad)
    ax.set_ylim(lo - pad, hi + pad)
    ax.set_box_aspect(1)
    ax.set_xlabel(f"Effect, {names[x_name]} (percentage points of GDP)")
    ax.set_ylabel(f"Effect, {names[y_names[0]] if len(y_names) == 1 else 'ridge'} (percentage points of GDP)")
    ax.set_title(title)
    ax.legend(loc="best", fontsize=8)
fig.tight_layout()
ctx.save_figure(fig, "case_estimator_comparison.png")

for x_name, y_names, _ in pairs:
    for y_name in y_names:
        both = att[[x_name, y_name]].dropna()
        print(
            f"{names[y_name]} against {names[x_name]}: {len(both)} episodes, correlation {np.corrcoef(both[x_name], both[y_name])[0, 1]:.3f}, "
            f"largest absolute difference {np.abs(both[y_name] - both[x_name]).max():.3f} percentage points."
        )

## Scope and limits

An effect is the mean difference between the observed outcome of the host economy and the outcome of a synthetic control over the post-opening window. It is a causal effect of the opening only if the synthetic control reproduces what the host economy would have shown without the opening. Anything else that changed in the host economy at about the same time, such as other investments, policy changes or exchange-rate movements, is part of the estimated effect. An episode that merges several openings measures their joint effect, and the effect of a single opening of such an episode is not identified.

An exact pre-opening fit is not a good fit. When the synthetic control reproduces every pre-opening year and the donors outnumber the pre-opening years, many weight vectors give the same pre-opening path and different post-opening paths, so the estimated effect is one point of a range. The table reports the smallest and the largest effect of that range, and the primary sample leaves such episodes out. The range covers only the weight vectors that are non-negative, sum to one and reproduce the pre-opening years within the tolerance of the settings cell. It is a statement about identification, not a confidence interval, and it does not include the sampling variation that the placebo standard deviation describes. The episodes set aside remain in the sensitivity sample of all feasible episodes.

The standard error is the standard deviation of the placebo effects. It describes how large effects are for donors that were not treated, and it is not the sampling error of a regression coefficient. The rank p-value is coarse: its smallest possible value is one divided by the number of donors plus one. The pre-opening fit criterion is relative to the placebo fits of the same episode, so the primary sample can contain episodes whose absolute pre-opening fit is poor when all of their placebos fit poorly as well.

The ridge variants allow negative weights and therefore extrapolate beyond the range of the donors. The horizon variants change the number of post-opening years but not the pre-opening fit. The difference-in-differences variant gives all donors the same weight and does not use the pre-opening fit. The feature tables use only years before the opening, and imputed values are medians of the reference table, so they carry no information about the effect. The estimates describe the episodes of the catalogue and the economies of the panel; they are not forecasts for another economy, which is the subject of notebooks 04 and 05.